# Task 3 (Medium) - Netflix Audience Rating Classification
**Auspify Technologies - Machine Learning Internship**

Task 3 (Medium) - Netflix Audience Rating Classification
Auspify Technologies - Machine Learning Internship

Predicts the audience rating category (e.g. TV-MA, PG-13, TV-Y7...) of a
Netflix title using its content attributes.

Workflow:

  Step 1: Analyze rating categories.
  
  Step 2: Prepare training datasets.
  
  Step 3: Train classification models.
  
  Step 4: Optimize model performance (hyperparameter tuning).
  
  Step 5: Evaluate prediction accuracy.

In [7]:
import re
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

DATA_PATH = "Dataset.csv"
RANDOM_STATE = 42
MIN_CLASS_COUNT = 10

# --- STEP 1: Analyze Ratings ---
def analyze_ratings(df: pd.DataFrame):
    counts = df["rating"].value_counts()
    print("Rating category distribution:")
    print(counts)
    rare = counts[counts < MIN_CLASS_COUNT].index.tolist()
    if rare:
        print(f"\nGrouping rare ratings into 'Other' (fewer than {MIN_CLASS_COUNT} samples): {rare}")
    return rare

# --- STEP 2: Prepare Dataset ---
def prepare_dataset(path: str):
    df = pd.read_csv(path)
    for col in ["director", "country", "listed_in", "rating", "duration"]:
        df[col] = df[col].fillna("")

    rare_ratings = analyze_ratings(df)
    df["rating_grouped"] = df["rating"].apply(lambda r: "Other" if r in rare_ratings else r)

    df["duration_value"] = df["duration"].apply(
        lambda x: int(re.search(r"\d+", x).group()) if re.search(r"\d+", x) else 0
    )
    df["genre_count"] = df["listed_in"].apply(lambda x: len(x.split(",")) if x else 0)
    df["primary_genre"] = df["listed_in"].apply(lambda x: x.split(",")[0].strip() if x else "Unknown")
    df["primary_country"] = df["country"].apply(lambda x: x.split(",")[0].strip() if x else "Unknown")

    features = df[[
        "type", "primary_genre", "primary_country",
        "release_year", "duration_value", "genre_count",
    ]].copy()

    for col in ["type", "primary_genre", "primary_country"]:
        features[col] = LabelEncoder().fit_transform(features[col])

    target_encoder = LabelEncoder()
    y = target_encoder.fit_transform(df["rating_grouped"])

    X_train, X_test, y_train, y_test = train_test_split(
        features, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )
    print(f"\nTrain size: {len(X_train)}, Test size: {len(X_test)}, Classes: {len(target_encoder.classes_)}")
    return X_train, X_test, y_train, y_test, target_encoder

# --- STEP 3: Train Baseline Models ---
def train_baseline_models(X_train, y_train):
    models = {
        "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
        "Random Forest": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    }
    for model in models.values():
        model.fit(X_train, y_train)
    return models

# --- STEP 4: Tune Random Forest ---
def tune_random_forest(X_train, y_train):
    param_grid = {
        "n_estimators": [100, 200],
        "max_depth": [None, 15, 25],
        "min_samples_leaf": [1, 2, 4],
    }
    grid = GridSearchCV(
        RandomForestClassifier(random_state=RANDOM_STATE),
        param_grid,
        cv=3,
        scoring="accuracy",
        n_jobs=-1,
    )
    grid.fit(X_train, y_train)
    print(f"\nBest params: {grid.best_params_}")
    print(f"Best CV accuracy: {grid.best_score_:.4f}")
    return grid.best_estimator_

# --- STEP 5: Evaluate Models ---
def evaluate(name, model, X_test, y_test, target_names):
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    f1m = f1_score(y_test, preds, average="macro")
    
    print(f"\n--- {name} ---")
    print(f"Accuracy: {acc:.4f}   F1 (macro): {f1m:.4f}")
    print(classification_report(y_test, preds, target_names=target_names, zero_division=0))
    print("Confusion matrix (rows=true, cols=pred):")
    print(pd.DataFrame(confusion_matrix(y_test, preds), index=target_names, columns=target_names))
    return acc, f1m

# --- MAIN WORKFLOW ---
def main():
    print("Step 1 & 2: Analyzing ratings and preparing training datasets...")
    X_train, X_test, y_train, y_test, target_encoder = prepare_dataset(DATA_PATH)

    # Baseline class majoritaire
    baseline_acc = pd.Series(y_train).value_counts(normalize=True).max()
    print(f"\nMajority-class baseline accuracy: {baseline_acc:.4f}")

    print("\nStep 3: Training baseline classification models...")
    baseline_models = train_baseline_models(X_train, y_train)
    baseline_results = {
        name: evaluate(name, model, X_test, y_test, target_encoder.classes_)
        for name, model in baseline_models.items()
    }

    print("\nStep 4: Optimizing Random Forest with GridSearchCV...")
    tuned_rf = tune_random_forest(X_train, y_train)
    tuned_acc, tuned_f1 = evaluate("Tuned Random Forest", tuned_rf, X_test, y_test, target_encoder.classes_)

    print("\nStep 5: Final accuracy comparison...")
    all_results = {**{k: v[0] for k, v in baseline_results.items()}, "Tuned Random Forest": tuned_acc}
    for name, acc in sorted(all_results.items(), key=lambda x: x[1], reverse=True):
        print(f"{name:22s}: {acc:.4f}")
    print(f"\nFor reference, majority-class baseline: {baseline_acc:.4f}")

if __name__ == "__main__":
    main()

Step 1 & 2: Analyzing ratings and preparing training datasets...
Rating category distribution:
rating
TV-MA       3205
TV-14       2157
TV-PG        861
R            799
PG-13        490
TV-Y7        333
TV-Y         306
PG           287
TV-G         220
NR            79
G             41
TV-Y7-FV       6
NC-17          3
UR             3
Name: count, dtype: int64

Grouping rare ratings into 'Other' (fewer than 10 samples): ['TV-Y7-FV', 'NC-17', 'UR']

Train size: 7032, Test size: 1758, Classes: 12

Majority-class baseline accuracy: 0.3646

Step 3: Training baseline classification models...

--- Decision Tree ---
Accuracy: 0.4346   F1 (macro): 0.3319
              precision    recall  f1-score   support

           G       0.67      0.50      0.57         8
          NR       0.00      0.00      0.00        16
       Other       0.00      0.00      0.00         2
          PG       0.53      0.52      0.52        58
       PG-13       0.25      0.28      0.26        98
           R     

## Conclusion
The tuned Random Forest reaches **53.1% accuracy** on 12 rating classes,
well above the 36.5% majority-class baseline. The confusion matrix shows
most errors are between *adjacent* maturity levels (e.g. TV-14 ↔ TV-MA,
TV-PG ↔ TV-14) rather than random mistakes, consistent with ratings being
an ordinal, somewhat fuzzy signal rather than a hard categorical one.